# 1. Section-Wise Statutory Corpus & Legal Sources Matrix
This compact prototype connects section-wise legal retrieval with deterministic document assembly and regression tests. Research context: the project owner’s early work at North South University covered problem definition, statutory data preparation and retrieval-pipeline design. The owner helped organise three Bangladeshi Acts by statutory section with parent Act metadata, then moved to another project while a sub-team continued the work. This recent notebook is a separate demonstration, not the original research implementation or a reproduction of the published system.

Fixed-size chunks can separate a condition or exception from the provision it qualifies. The intended boundary is a complete statutory section with `[Act Name - Section ID]` metadata. **For portability, these ten records contain educational section summaries, not complete official provisions.** Replace them with verified full text and provisos from https://bdlaws.minlaw.gov.bd before real use. Statutory classifications and legal applicability require professional checking.

`active`/`amended` are simulated source-registry states. Two synthetic High Court interpretation updates are included; they do not assert actual amendments or real recent judgments. All examples are fictional. Run all cells in order on CPU; Python, NumPy, pandas and scikit-learn suffice, without credentials or network calls.

Source traceability also requires jurisdiction, official-source and version metadata in production: Act names can overlap between Bangladesh and India. Those production metadata checks are not implemented here.


In [1]:
from copy import deepcopy
from datetime import date
import hashlib, json, re
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def sha(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True).encode()).hexdigest()

rows = [
    ('C54','Code of Criminal Procedure, 1898','54','Arrest without warrant requires examination of statutory grounds and safeguards.'),
    ('C167','Code of Criminal Procedure, 1898','167','Further detention during an incomplete investigation requires Magistrate authorization subject to statutory limits.'),
    ('C496','Code of Criminal Procedure, 1898','496','A person accused of a bailable offence who is prepared to give bail shall be released, subject to the section.'),
    ('C497','Code of Criminal Procedure, 1898','497','Bail for non bailable offences is subject to statutory restrictions, exceptions and further inquiry provisions.'),
    ('C417','Code of Criminal Procedure, 1898','417','Appeal against acquittal follows statutory requirements concerning the applicant, forum, permission and time limits.'),
    ('C439','Code of Criminal Procedure, 1898','439','The High Court Division exercises specified revision powers subject to procedural protections and statutory restrictions.'),
    ('P379','Penal Code, 1860','379','Theft is punishable with imprisonment up to three years, or fine, or both.'),
    ('P420','Penal Code, 1860','420','Cheating and dishonestly inducing delivery of property is punishable with imprisonment up to seven years and fine.'),
    ('S3','Special Powers Act, 1974','3','Preventive detention orders depend on the statutory authority and conditions concerning prejudicial acts.'),
    ('S8','Special Powers Act, 1974','8','Grounds of detention must be communicated within the statutory framework, allowing representation subject to the section.')
]
LEGAL_SOURCES_MATRIX = [dict(source_id=s, act_name=a, section_number=n,
    prepared_text=f'[{a} - Section {n}] {t}', status='active', recent_judgment_note=None)
    for s,a,n,t in rows]
for source in LEGAL_SOURCES_MATRIX[:2]:
    source.update(status='amended', recent_judgment_note=dict(
        id='SIM_HCD_'+source['source_id'], date='2026-01-15', simulated=True,
        text='Synthetic interpretation update: review arrest or detention safeguards.'))
display(pd.DataFrame(LEGAL_SOURCES_MATRIX))

,source_id,act_name,section_number,prepared_text,status,recent_judgment_note
0,C54,"Code of Criminal Procedure, 1898",54,"[Code of Criminal Procedure, 1898 - Section 54...",amended,"{'id': 'SIM_HCD_C54', 'date': '2026-01-15', 's..."
1,C167,"Code of Criminal Procedure, 1898",167,"[Code of Criminal Procedure, 1898 - Section 16...",amended,"{'id': 'SIM_HCD_C167', 'date': '2026-01-15', '..."
2,C496,"Code of Criminal Procedure, 1898",496,"[Code of Criminal Procedure, 1898 - Section 49...",active,None
3,C497,"Code of Criminal Procedure, 1898",497,"[Code of Criminal Procedure, 1898 - Section 49...",active,None
4,C417,"Code of Criminal Procedure, 1898",417,"[Code of Criminal Procedure, 1898 - Section 41...",active,None
5,C439,"Code of Criminal Procedure, 1898",439,"[Code of Criminal Procedure, 1898 - Section 43...",active,None
6,P379,"Penal Code, 1860",379,"[Penal Code, 1860 - Section 379] Theft is puni...",active,None
7,P420,"Penal Code, 1860",420,"[Penal Code, 1860 - Section 420] Cheating and ...",active,None
8,S3,"Special Powers Act, 1974",3,"[Special Powers Act, 1974 - Section 3] Prevent...",active,None
9,S8,"Special Powers Act, 1974",8,"[Special Powers Act, 1974 - Section 8] Grounds...",active,None


# 2. Versioned Knowledge Base of Modular Pleading Components
Each block has exact fact conditions, prerequisite blocks, configured conflicts, an order and required citations. Version 1.2.0 identifies the rules; a content fingerprint detects changes even if the version label is accidentally reused. The procedural limitation flag represents a hypothetical counsel-verified position, not an automated legal conclusion. Exclusive routes are teaching policy, not a statement that legal arguments cannot coexist.

In [2]:
def block(mid, order, facts, deps, conflicts, statutes, text):
    return dict(id=mid, order_index=order, required_facts=facts, depends_on=deps,
                mutually_exclusive_with=conflicts, linked_statutes=statutes, template_text=text)

KNOWLEDGE_BASE_V1_2 = dict(version='1.2.0', as_of='2025-12-01', modules=[
    block('M1_JURISDICTION',10,{'forum':'magistrate'},[],[],[],
          'JURISDICTION: Application to the competent Magistrate; competence requires verification.'),
    block('M2_FACTUAL_SUMMARY',20,{'has_facts':True},['M1_JURISDICTION'],[],[],
          'FACTS: {client_name}, case {case_id}: {summary}'),
    block('M3_PROCEDURAL_DEFENSE_LIMITATION',30,{'limitation_defense':True},['M2_FACTUAL_SUMMARY'],
          ['M4_SUBSTANTIVE_STATUTORY_GROUNDS'],['C54','C167'],
          'PROCEDURE: Counsel requests examination of detention limits and safeguards [C54] [C167].'),
    block('M4_SUBSTANTIVE_STATUTORY_GROUNDS',40,{'substantive_defense':True},['M2_FACTUAL_SUMMARY'],
          ['M3_PROCEDURAL_DEFENSE_LIMITATION'],['P420'],
          'GROUNDS: Counsel disputes the alleged ingredients of the charged offence [P420].'),
    block('M5_EVIDENTIARY_CHALLENGE',50,{'has_records':True},['M3_PROCEDURAL_DEFENSE_LIMITATION'],[],['C167'],
          'RECORDS: Seek examination of the custody timeline and authorization record [C167].'),
    block('M6_ALTERNATIVE_RELIEF',60,{'seek_bail':True},['M2_FACTUAL_SUMMARY'],[],['C497'],
          'ALTERNATIVE RELIEF: Request consideration of bail subject to applicable restrictions [C497].'),
    block('M7_PRAYER_AND_RELIEF',70,{'request_relief':True},['M2_FACTUAL_SUMMARY'],[],[],
          'PRAYER: Request an appropriate reasoned order after review of the record and law.')])
display(pd.DataFrame(KNOWLEDGE_BASE_V1_2['modules']).drop(columns='template_text'))

,id,order_index,required_facts,depends_on,mutually_exclusive_with,linked_statutes
0,M1_JURISDICTION,10,{'forum': 'magistrate'},[],[],[]
1,M2_FACTUAL_SUMMARY,20,{'has_facts': True},[M1_JURISDICTION],[],[]
2,M3_PROCEDURAL_DEFENSE_LIMITATION,30,{'limitation_defense': True},[M2_FACTUAL_SUMMARY],[M4_SUBSTANTIVE_STATUTORY_GROUNDS],"[C54, C167]"
3,M4_SUBSTANTIVE_STATUTORY_GROUNDS,40,{'substantive_defense': True},[M2_FACTUAL_SUMMARY],[M3_PROCEDURAL_DEFENSE_LIMITATION],[P420]
4,M5_EVIDENTIARY_CHALLENGE,50,{'has_records': True},[M3_PROCEDURAL_DEFENSE_LIMITATION],[],[C167]
5,M6_ALTERNATIVE_RELIEF,60,{'seek_bail': True},[M2_FACTUAL_SUMMARY],[],[C497]
6,M7_PRAYER_AND_RELIEF,70,{'request_relief': True},[M2_FACTUAL_SUMMARY],[],[]


# 3. Deterministic Assembly Engine (Zero LLM Strategy)
Facts and rules alone control assembly. No model selects strategy or rewrites facts. Every block receives an INCLUDED or SKIPPED decision with predicate, dependency and conflict details. Invalid facts, malformed dependency ordering and active route conflicts raise errors before drafting. Text hashes verify repeatability; the KB fingerprint identifies the rule snapshot.

In [3]:
class ConflictError(ValueError): pass

class DeterministicAssemblyEngine:
    def __init__(self, kb):
        self.kb = deepcopy(kb)
        self.modules = sorted(self.kb['modules'], key=lambda m:(m['order_index'],m['id']))
        positions = {m['id']:i for i,m in enumerate(self.modules)}
        if len(positions) != len(self.modules): raise ValueError('Duplicate module ID')
        for m in self.modules:
            if any(d not in positions or positions[d]>=positions[m['id']] for d in m['depends_on']):
                raise ValueError('Missing, cyclic or incorrectly ordered dependency')
            if any(c not in positions or c==m['id'] for c in m['mutually_exclusive_with']):
                raise ValueError('Invalid conflict reference')

    def assemble(self, case_facts):
        f = deepcopy(case_facts)
        bools = ['has_facts','limitation_defense','substantive_defense','has_records','seek_bail','request_relief']
        if set(f)-set(bools+['case_id','client_name','summary','forum']): raise ValueError('Unknown fact')
        for key in ['case_id','client_name','summary','forum']:
            if not isinstance(f.get(key),str) or not f[key].strip(): raise ValueError('Missing string: '+key)
        if f['forum'] not in ['magistrate','high_court']: raise ValueError('Unsupported forum')
        for key in bools:
            f.setdefault(key,False)
            if type(f[key]) is not bool: raise ValueError('Expected Boolean: '+key)
        chosen, trace = [], []
        for m in self.modules:
            rules = {k:dict(expected=v,actual=f[k],passed=f[k]==v) for k,v in m['required_facts'].items()}
            missing = [d for d in m['depends_on'] if d not in chosen]
            include = all(r['passed'] for r in rules.values()) and not missing
            if include: chosen.append(m['id'])
            trace.append(dict(module=m['id'],decision='INCLUDED' if include else 'SKIPPED',
                rules=rules,missing_dependencies=missing,linked_statutes=m['linked_statutes'],
                kb_version=self.kb['version'],kb_fingerprint=sha(self.kb)))
        for m,t in zip(self.modules,trace):
            t['active_conflicts'] = sorted(c for c in m['mutually_exclusive_with'] if c in chosen and m['id'] in chosen)
        conflicts = sorted({tuple(sorted([t['module'],c])) for t in trace for c in t['active_conflicts']})
        if conflicts: raise ConflictError('Exclusive routes: '+repr(conflicts))
        text = '\n\n'.join(m['template_text'].format(**f) for m in self.modules if m['id'] in chosen)
        for t in trace: t['output_sha256'] = sha(text)
        return dict(text=text,module_ids=chosen,linked_statutes=sorted({s for t in trace
            if t['decision']=='INCLUDED' for s in t['linked_statutes']}),audit_trace=trace,
            output_sha256=sha(text),kb_version=self.kb['version'],kb_fingerprint=sha(self.kb))

engine = DeterministicAssemblyEngine(KNOWLEDGE_BASE_V1_2)
SAMPLE_CASE = dict(case_id='DEMO-1',client_name='Amina (fictional)',forum='magistrate',
    summary='Counsel seeks review of arrest and custody records.',has_facts=True,
    limitation_defense=True,has_records=True,request_relief=True)
assembled_result = engine.assemble(SAMPLE_CASE)
print(assembled_result['text'])
display(pd.DataFrame(assembled_result['audit_trace']))

JURISDICTION: Application to the competent Magistrate; competence requires verification.

FACTS: Amina (fictional), case DEMO-1: Counsel seeks review of arrest and custody records.

PROCEDURE: Counsel requests examination of detention limits and safeguards [C54] [C167].

RECORDS: Seek examination of the custody timeline and authorization record [C167].

PRAYER: Request an appropriate reasoned order after review of the record and law.


,module,decision,rules,missing_dependencies,linked_statutes,kb_version,kb_fingerprint,active_conflicts,output_sha256
0,M1_JURISDICTION,INCLUDED,"{'forum': {'expected': 'magistrate', 'actual':...",[],[],1.2.0,02405c52e5b75f30501f7194f916e9467d8a957a54632d...,[],42ae80a2121d8cafcd4515e86de5790954b8d7cee46b01...
1,M2_FACTUAL_SUMMARY,INCLUDED,"{'has_facts': {'expected': True, 'actual': Tru...",[],[],1.2.0,02405c52e5b75f30501f7194f916e9467d8a957a54632d...,[],42ae80a2121d8cafcd4515e86de5790954b8d7cee46b01...
2,M3_PROCEDURAL_DEFENSE_LIMITATION,INCLUDED,"{'limitation_defense': {'expected': True, 'act...",[],"[C54, C167]",1.2.0,02405c52e5b75f30501f7194f916e9467d8a957a54632d...,[],42ae80a2121d8cafcd4515e86de5790954b8d7cee46b01...
3,M4_SUBSTANTIVE_STATUTORY_GROUNDS,SKIPPED,"{'substantive_defense': {'expected': True, 'ac...",[],[P420],1.2.0,02405c52e5b75f30501f7194f916e9467d8a957a54632d...,[],42ae80a2121d8cafcd4515e86de5790954b8d7cee46b01...
4,M5_EVIDENTIARY_CHALLENGE,INCLUDED,"{'has_records': {'expected': True, 'actual': T...",[],[C167],1.2.0,02405c52e5b75f30501f7194f916e9467d8a957a54632d...,[],42ae80a2121d8cafcd4515e86de5790954b8d7cee46b01...
5,M6_ALTERNATIVE_RELIEF,SKIPPED,"{'seek_bail': {'expected': True, 'actual': Fal...",[],[C497],1.2.0,02405c52e5b75f30501f7194f916e9467d8a957a54632d...,[],42ae80a2121d8cafcd4515e86de5790954b8d7cee46b01...
6,M7_PRAYER_AND_RELIEF,INCLUDED,"{'request_relief': {'expected': True, 'actual'...",[],[],1.2.0,02405c52e5b75f30501f7194f916e9467d8a957a54632d...,[],42ae80a2121d8cafcd4515e86de5790954b8d7cee46b01...


# 4. Threshold-Gated Retrieval & Source Matrix Review
The local layer retrieves source context and reviews citations. It is **not complete generative RAG**: no language model generates an answer from retrieved sections. Drafting and retrieval are separate paths. Sparse TF-IDF cosine scores are not confidence or legal accuracy. The later research pipeline used a similarity threshold to trigger query decomposition. The project owner did not establish 0.80 as the optimal threshold. Here, `tau=0.80` is an illustrative value from the reference approach, uncalibrated for this TF-IDF index. Related context can still be incomplete; a high similarity score does not guarantee that a passage answers every part of a question.

If direct retrieval fails the gate, a bounded rule splits explicit semicolons, or `and` when no semicolon exists. This simulates the role of query decomposition, not Llama inference. Results are deduplicated by source, ranked by score, and interleaved across issues for coverage. Every returned section must still clear the gate; unresolved issues abstain. Exact-record queries make the demonstration reproducible, not representative of paraphrase performance.

The review derives mandatory citations from the trusted module definitions, not just output metadata. It checks text omissions, unknown matrix references and newer synthetic judgment notes. Suggested sources are search leads for professional review, not declarations of applicability.

In [4]:
class LegalRAGReviewLayer:
    def __init__(self, sources, kb):
        self.sources, self.kb = deepcopy(sources), deepcopy(kb)
        self.by_id = {s['source_id']:s for s in self.sources}
        if len(self.by_id)!=len(self.sources): raise ValueError('Duplicate source ID')
        self.vectorizer = TfidfVectorizer(ngram_range=(1,2))
        self.index = self.vectorizer.fit_transform([s['prepared_text'] for s in self.sources])

    def _search(self, query, tau):
        scores = cosine_similarity(self.vectorizer.transform([query]),self.index)[0]
        ranked = sorted(range(len(scores)),key=lambda i:(-scores[i],self.sources[i]['source_id']))
        hits = [dict(source_id=self.sources[i]['source_id'],score=float(scores[i]),
                     context=self.sources[i]['prepared_text']) for i in ranked if scores[i]>=tau]
        return dict(query=query,top_score=float(np.max(scores)),hits=hits,abstained=not bool(hits))

    def retrieve_with_decomposition(self, query, tau=0.80):
        if not isinstance(query,str) or not query.strip() or not 0<tau<=1: raise ValueError('Invalid query or threshold')
        direct = self._search(query,tau)
        parts = [p.strip() for p in (query.split(';') if ';' in query else re.split(r'\s+and\s+',query)) if p.strip()]
        fallback = [self._search(p,tau) for p in parts] if direct['abstained'] and len(parts)>1 else []
        merged, seen = [], set()
        groups = [r['hits'] for r in (fallback or [direct])]
        for rank in range(max(map(len,groups),default=0)):
            for group in groups:
                if rank<len(group) and group[rank]['source_id'] not in seen:
                    merged.append(group[rank]); seen.add(group[rank]['source_id'])
        return dict(mode='decomposed' if fallback else 'direct',direct=direct,subqueries=fallback,
                    hits=merged,unresolved=[r['query'] for r in (fallback or [direct]) if r['abstained']])

    def review_assembled_pleading(self, result, case_facts):
        expected = DeterministicAssemblyEngine(self.kb).assemble(case_facts)
        if result['module_ids']!=expected['module_ids'] or result['kb_fingerprint']!=sha(self.kb):
            raise ValueError('Facts or KB do not match assembly')
        citations = set(re.findall(r'\[([CPS]\d+)\]',result['text']))
        required = set(expected['linked_statutes'])
        alerts = []
        for m in self.kb['modules']:
            if m['id'] not in result['module_ids']: continue
            for sid in m['linked_statutes']:
                if sid not in self.by_id or sid not in citations:
                    alerts.append(dict(flag='[MISSING CITATION CHECK]',module=m['id'],source_id=sid))
                note = self.by_id.get(sid,{}).get('recent_judgment_note')
                if note and date.fromisoformat(note['date'])>date.fromisoformat(self.kb['as_of']):
                    alerts.append(dict(flag='[JUDGMENT IMPACT WARNING]',module=m['id'],source_id=sid,note=note))
        for sid in sorted(citations-required | citations-set(self.by_id)):
            alerts.append(dict(flag='[MISSING CITATION CHECK]',source_id=sid,reason='Unlinked or unknown text citation'))
        suggestions = self.retrieve_with_decomposition(case_facts['summary'])
        return dict(alerts=alerts,suggested_relevant_sources={'flag':'[SUGGESTED RELEVANT SOURCES]',
                    'hits':suggestions['hits'],'unresolved':suggestions['unresolved']})

review = LegalRAGReviewLayer(LEGAL_SOURCES_MATRIX,KNOWLEDGE_BASE_V1_2)
direct_demo = review.retrieve_with_decomposition(LEGAL_SOURCES_MATRIX[0]['prepared_text'])
compound_query = LEGAL_SOURCES_MATRIX[0]['prepared_text']+'; '+LEGAL_SOURCES_MATRIX[7]['prepared_text']
fallback_demo = review.retrieve_with_decomposition(compound_query)
display(pd.DataFrame([dict(mode=r['mode'],initial_score=r['direct']['top_score'],
    retrieved=[h['source_id'] for h in r['hits']],unresolved=r['unresolved']) for r in [direct_demo,fallback_demo]]))
print(json.dumps(review.review_assembled_pleading(assembled_result,SAMPLE_CASE),indent=2))

,mode,initial_score,retrieved,unresolved
0,direct,1.000000,[C54],[]
1,decomposed,0.787877,"[C54, P420]",[]


{
  "alerts": [
    {
      "flag": "[JUDGMENT IMPACT WARNING]",
      "module": "M3_PROCEDURAL_DEFENSE_LIMITATION",
      "source_id": "C54",
      "note": {
        "id": "SIM_HCD_C54",
        "date": "2026-01-15",
        "simulated": true,
        "text": "Synthetic interpretation update: review arrest or detention safeguards."
      }
    },
    {
      "flag": "[JUDGMENT IMPACT WARNING]",
      "module": "M3_PROCEDURAL_DEFENSE_LIMITATION",
      "source_id": "C167",
      "note": {
        "id": "SIM_HCD_C167",
        "date": "2026-01-15",
        "simulated": true,
        "text": "Synthetic interpretation update: review arrest or detention safeguards."
      }
    },
    {
      "flag": "[JUDGMENT IMPACT WARNING]",
      "module": "M5_EVIDENTIARY_CHALLENGE",
      "source_id": "C167",
      "note": {
        "id": "SIM_HCD_C167",
        "date": "2026-01-15",
        "simulated": true,
        "text": "Synthetic interpretation update: review arrest or detention safeguards."
 

# 5. Automated Regression Tests on Reference Cases
A maintained legal knowledge base should be checked against de-identified, lawyer-approved historical records. **None were supplied here.** `HISTORICAL_APPROVED_CASES` preserves the requested interface name, but contains four fictional fixtures with `approval_status='simulated_not_approved'`. Their expected outputs are written explicitly rather than generated by the engine. Passing parity means fixture parity, not historical legal validation.

The tests detect nondeterministic output, changed block sequences or citations, lost conflict enforcement and broken judgment propagation. Additional negative checks verify missing citations and unknown source IDs. A new judgment is injected into a clean source copy; the assertion verifies the exact affected blocks. Assertions raise if any test fails; exceptions are reported honestly before the final failure check.

<details><summary>Technical reference</summary>

Adnan, Khurshed, Sharifeen and Momen, *JuristRAG: An Enhanced RAG System for Semantic Question Answering on Legal Statutes*, ICCIT 2025, Section III.A–C: section-wise preprocessing, Act metadata and threshold-gated query decomposition. This notebook uses sparse TF-IDF and rule-based splitting instead of BGE-M3, ChromaDB and Llama. It does not reproduce the paper's benchmark results.

</details>

In [5]:
HISTORICAL_APPROVED_CASES = [
    dict(id='H1',approval_status='simulated_not_approved',case_facts=SAMPLE_CASE,
         modules=['M1_JURISDICTION','M2_FACTUAL_SUMMARY','M3_PROCEDURAL_DEFENSE_LIMITATION','M5_EVIDENTIARY_CHALLENGE','M7_PRAYER_AND_RELIEF'],citations=['C167','C54']),
    dict(id='H2',approval_status='simulated_not_approved',case_facts=dict(SAMPLE_CASE,limitation_defense=False,substantive_defense=True),
         modules=['M1_JURISDICTION','M2_FACTUAL_SUMMARY','M4_SUBSTANTIVE_STATUTORY_GROUNDS','M7_PRAYER_AND_RELIEF'],citations=['P420']),
    dict(id='H3',approval_status='simulated_not_approved',case_facts=dict(SAMPLE_CASE,limitation_defense=False,seek_bail=True),
         modules=['M1_JURISDICTION','M2_FACTUAL_SUMMARY','M6_ALTERNATIVE_RELIEF','M7_PRAYER_AND_RELIEF'],citations=['C497']),
    dict(id='H4',approval_status='simulated_not_approved',case_facts=dict(SAMPLE_CASE,has_facts=False),
         modules=['M1_JURISDICTION'],citations=[])]

def test_deterministic_reproducibility():
    outputs = [engine.assemble(SAMPLE_CASE) for _ in range(25)]
    assert len({r['output_sha256'] for r in outputs})==1 and len({sha(r) for r in outputs})==1

def test_historical_approved_cases():
    for fixture in HISTORICAL_APPROVED_CASES:
        out = engine.assemble(fixture['case_facts'])
        assert out['module_ids']==fixture['modules'] and out['linked_statutes']==fixture['citations']
        assert sorted(set(re.findall(r'\[([CPS]\d+)\]',out['text'])))==fixture['citations']

def test_mutual_exclusion_guard():
    messages = []
    for _ in range(2):
        try: engine.assemble(dict(SAMPLE_CASE,substantive_defense=True))
        except ConflictError as exc: messages.append(str(exc))
        else: raise AssertionError('Conflict was accepted')
    assert messages[0]==messages[1]

def test_new_judgment_impact_flagging():
    sources = deepcopy(LEGAL_SOURCES_MATRIX)
    for s in sources: s['recent_judgment_note']=None
    assert not LegalRAGReviewLayer(sources,KNOWLEDGE_BASE_V1_2).review_assembled_pleading(assembled_result,SAMPLE_CASE)['alerts']
    sources[1]['recent_judgment_note']=dict(id='SIM_NEW',date='2026-03-01',simulated=True,text='Synthetic new remand interpretation')
    layer = LegalRAGReviewLayer(sources,KNOWLEDGE_BASE_V1_2)
    alerts = layer.review_assembled_pleading(assembled_result,SAMPLE_CASE)['alerts']
    assert {a['module'] for a in alerts}=={'M3_PROCEDURAL_DEFENSE_LIMITATION','M5_EVIDENTIARY_CHALLENGE'}
    assert all(a['source_id']=='C167' and a['flag']=='[JUDGMENT IMPACT WARNING]' for a in alerts)
    broken = dict(assembled_result,text=assembled_result['text'].replace('[C54]','')+' [C999]')
    checks = layer.review_assembled_pleading(broken,SAMPLE_CASE)['alerts']
    assert {a['source_id'] for a in checks if a['flag']=='[MISSING CITATION CHECK]'}=={'C54','C999'}
    assert direct_demo['mode']=='direct' and fallback_demo['mode']=='decomposed'
    assert {h['source_id'] for h in fallback_demo['hits']}=={'C54','P420'}
    assert not review.retrieve_with_decomposition('neutrino spectroscopy')['hits']

results = []
for test in [test_deterministic_reproducibility,test_historical_approved_cases,
             test_mutual_exclusion_guard,test_new_judgment_impact_flagging]:
    try:
        test(); results.append(dict(test=test.__name__,status='PASS',detail='Assertions completed'))
    except Exception as exc: results.append(dict(test=test.__name__,status='FAIL',detail=repr(exc)))
display(pd.DataFrame(results))
assert all(r['status']=='PASS' for r in results), 'Regression failure; see summary'

,test,status,detail
0,test_deterministic_reproducibility,PASS,Assertions completed
1,test_historical_approved_cases,PASS,Assertions completed
2,test_mutual_exclusion_guard,PASS,Assertions completed
3,test_new_judgment_impact_flagging,PASS,Assertions completed
